In [252]:
# --- Работа с данными ---
import pandas as pd
import numpy as np

# --- Визуализация ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Scikit-Learn: Модели и разделение данных ---
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# --- Scikit-Learn: Метрики ---
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    mean_absolute_error, 
    mean_absolute_percentage_error, 
    mean_squared_error)

# --- Глубокое обучение (PyTorch) ---
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchsummary import summary
from sklearn.linear_model import Ridge


from sklearn.pipeline import Pipeline

import keras
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping

In [31]:
dataset = pd.read_csv('C:/Users/Pavel/Lessen_jypyter/csv/cardiovascular_risk_dataset.csv')

In [32]:
dataset

,Patient_ID,age,bmi,systolic_bp,diastolic_bp,cholesterol_mg_dl,resting_heart_rate,smoking_status,daily_steps,stress_level,physical_activity_hours_per_week,sleep_hours,family_history_heart_disease,diet_quality_score,alcohol_units_per_week,heart_disease_risk_score,risk_category
0,1,62,25.0,142,93,247,72,Never,11565,3,5.6,8.2,No,7,0.7,28.1,Medium
1,2,54,29.7,158,101,254,74,Current,4036,8,0.5,6.7,No,5,4.5,63.0,High
2,3,46,36.2,170,113,276,80,Current,3043,9,0.4,4.0,No,1,20.8,73.1,High
3,4,48,30.4,153,98,230,73,Former,5604,5,0.6,8.0,No,4,8.5,39.5,Medium
4,5,46,25.3,139,87,206,69,Current,7464,1,2.0,6.1,No,5,3.6,29.3,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,5496,19,26.0,121,75,185,84,Never,6724,3,2.9,7.2,No,7,0.0,0.0,Low
5496,5497,18,30.9,128,82,235,75,Never,3661,4,0.0,5.5,No,1,9.6,16.8,Low
5497,5498,63,29.5,142,92,239,69,Never,6643,5,4.1,6.9,No,6,2.4,31.8,Medium
5498,5499,46,27.5,138,91,237,65,Never,3279,3,2.4,5.8,Yes,5,2.3,29.4,Medium


In [ ]:
dataset['smoking_status'] = dataset['smoking_status'].map({'Never': 2,'Former': 1, 'Current': 0}).astype(float)
dataset['family_history_heart_disease'] = dataset['family_history_heart_disease'].map({'Yes': 1, 'No': 0}).astype(float)
dataset['smoking_status'] = dataset['smoking_status'].fillna(dataset['smoking_status'].mean())
dataset = dataset.drop(['Patient_ID', 'risk_category'], axis = 1)

In [6]:
col = dataset.columns
for i in dataset[col]:
    dataset[i] = dataset[i].astype(float)

In [34]:
dataset

,age,bmi,systolic_bp,diastolic_bp,cholesterol_mg_dl,resting_heart_rate,smoking_status,daily_steps,stress_level,physical_activity_hours_per_week,sleep_hours,family_history_heart_disease,diet_quality_score,alcohol_units_per_week,heart_disease_risk_score
0,62,25.0,142,93,247,72,2.0,11565,3,5.6,8.2,0.0,7,0.7,28.1
1,54,29.7,158,101,254,74,0.0,4036,8,0.5,6.7,0.0,5,4.5,63.0
2,46,36.2,170,113,276,80,0.0,3043,9,0.4,4.0,0.0,1,20.8,73.1
3,48,30.4,153,98,230,73,1.0,5604,5,0.6,8.0,0.0,4,8.5,39.5
4,46,25.3,139,87,206,69,0.0,7464,1,2.0,6.1,0.0,5,3.6,29.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,19,26.0,121,75,185,84,2.0,6724,3,2.9,7.2,0.0,7,0.0,0.0
5496,18,30.9,128,82,235,75,2.0,3661,4,0.0,5.5,0.0,1,9.6,16.8
5497,63,29.5,142,92,239,69,2.0,6643,5,4.1,6.9,0.0,6,2.4,31.8
5498,46,27.5,138,91,237,65,2.0,3279,3,2.4,5.8,1.0,5,2.3,29.4


In [7]:
dataset.dtypes

age                                 float64
bmi                                 float64
systolic_bp                         float64
diastolic_bp                        float64
cholesterol_mg_dl                   float64
resting_heart_rate                  float64
smoking_status                      float64
daily_steps                         float64
stress_level                        float64
physical_activity_hours_per_week    float64
sleep_hours                         float64
family_history_heart_disease        float64
diet_quality_score                  float64
alcohol_units_per_week              float64
risk_category                       float64
dtype: object

In [182]:
def split(dataset):
    target = dataset['heart_disease_risk_score']
    features = dataset.drop(['heart_disease_risk_score'], axis = 1)
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = .2, random_state = 42)
    
    incol = X_train.shape[1]
    return X_train, X_test, y_train, y_test, incol

x_train, x_test, y_train, y_test, incol = split(dataset)

In [214]:
class split_data():
    def __init__(self,feature_train, feature_test, target_train, target_test, incol, batch = 64):
        self.feature_train = torch.FloatTensor(feature_train.values)
        self.feature_test = torch.LongTensor(feature_test.values)
        self.target_train = torch.FloatTensor(target_train.values)
        self.target_test = torch.LongTensor(target_test.values)
        self.incol = incol
        self.batch = batch
        
    def train_test(self):
        train = DataLoader(
            TensorDataset(self.feature_train, self.target_train),
            batch_size = self.batch,
            shuffle = True
        )
        test = DataLoader(
            TensorDataset(self.feature_test, self.target_test),
            batch_size = self.batch,
            shuffle = False
        )
        return train, test, self.incol

tts = split_data(x_train,x_test,y_train,y_test,incol)
trainloader , testloader , incol= tts.train_test()

In [184]:
def model_keras(feature_train, feature_test, target_train, target_test, incol):
    modelKeras = keras.Sequential([
        keras.layers.Dense(128, input_shape = (feature_train.shape[1],)),
        keras.layers.Dense(64, activation = LeakyReLU(alpha = 0.01)),
            keras.layers.Dropout(0.2),
            keras.layers.BatchNormalization(),
        keras.layers.Dense(32, activation = 'relu', kernel_regularizer='l2'),
            keras.layers.Dropout(0.2),
            keras.layers.BatchNormalization(),
        keras.layers.Dense(1)
    ])

    modelKeras.compile(
            optimizer = keras.optimizers.Adam(learning_rate = 0.01),
            loss = 'mse',
            metrics = ['mae']
    )
    early_stopping = EarlyStopping(
            monitor='val_loss',
            min_delta=0.01,
            patience=10,
            verbose=0,
            baseline=None,
            restore_best_weights=True,
            start_from_epoch=0
    )
    batch = 64
    modelKeras.fit(feature_train, target_train,batch_size = batch, epochs = 200, callbacks = [early_stopping], validation_split = 0.1,verbose = 0)

    train_loss,train_mae = modelKeras.evaluate(feature_train,target_train)
    test_loss,test_mae = modelKeras.evaluate(feature_test,target_test)
    print(f"Результат Тренировочной и тестовой оценки : {train_mae} / {test_mae}")

In [255]:
class modelTorch(nn.Module):
    def __init__(self,input_size = incol,
                hidden_size1 = 128,
                hidden_size2 = 64,
                hidden_size3 = 32,
                output_size = 1):
        super(modelTorch, self).__init__()
        self.hidden1 = nn.Linear(input_size, hidden_size1)
        self.hidden2 = nn.Linear(hidden_size1, hidden_size2)
        self.activation1 = nn.LeakyReLU(0.1)
        self.drop1 = nn.Dropout(0.2)
        self.norm = nn.BatchNorm1d(hidden_size2)
        self.hidden3 = nn.Linear(hidden_size2, hidden_size3)
        self.activation2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.2)
        self.norm2 = nn.BatchNorm1d(hidden_size3)
        self.outp = nn.Linear(hidden_size3, output_size)


    def forward(self, val):
        out = self.hidden1(val)
        out = self.hidden2(out)
        out = self.activation1(out)
        out = self.drop1(out)
        out = self.norm(out)
        out = self.hidden3(out)
        out = self.activation2(out)
        out = self.drop2(out)
        out = self.norm2(out)
        out = self.outp(out)
        return out

    

In [276]:
ModelTorch = modelTorch()
lert = 0.01
loss =nn.MSELoss()
optimizer = torch.optim.Adam(ModelTorch.parameters(), lr = lert, weight_decay=0.001)

In [270]:
def train(loader, model, sloss, optim):
    model.train()
    step = len(loader)
    nepoch = 5
    for epoch in range(nepoch):
        for i, (batch, label) in enumerate(loader):
            outputs = model(batch)
            if label.dim() == 1:
                label = label.unsqueeze(1).float() 
            elif label.dim() == 2 and label.dtype != torch.float32:
                label = label.float()
            loss = sloss(outputs, label)
            
            optim.zero_grad()
            loss.backward()
            optim.step()

        print('Epoch [{}/{}], loss:{:.4f}'.format(epoch+1, nepoch, loss.item()))

In [273]:
def test(loader, model, sloss):
    model.eval()
    size = len(loader.dataset)
    nbatches = len(loader)
    test_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.float()
            pred = model(x)
            y = y.unsqueeze(1)
            test_loss += sloss(pred, y).item()
    loss = test_loss / nbatches
    print(f"MSE: {loss}")

In [272]:
train(trainloader, ModelTorch, loss, optimizer)

Epoch [1/5], loss:41.6694
Epoch [2/5], loss:45.4845
Epoch [3/5], loss:33.2149
Epoch [4/5], loss:47.4289
Epoch [5/5], loss:32.3855


In [274]:
test(testloader, ModelTorch, loss)

MSE: 48.536930878957115


In [185]:
model_keras(x_train,x_test,y_train,y_test,incol)

C:\Users\Pavel\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Pavel\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 27.4990 - mae: 4.1512 
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28.7604 - mae: 4.2326 
Результат Тренировочной и тестовой оценки : 4.151227951049805 / 4.232577800750732
